In [ ]:
!date

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import scanpy as sc
import anndata

In [ ]:
sc.set_figure_params(figsize=(5,5), frameon=False)
sc.settings.verbosity = 3

In [ ]:
projdir = '/u/project/cluo/terencew/igvf/2023_YR2/multiome/'
donors = list(np.loadtxt(f'{projdir}/txt/donors.txt', dtype=str))

tmp_order = ['Start', 'Sendai', 'Delayed', 'Fail1', 'Inter1', 'Inter2', 'Fail2', 'IPS']

In [ ]:
adata = sc.read_h5ad(f'{projdir}/h5ad/rna/raw/intersect_atac.h5ad')

In [ ]:
adata

In [ ]:
tmp_adata = sc.read_h5ad(f'{projdir}/h5ad/rna/processed/intersect_atac_reindex_small.h5ad')
adata.obsm = tmp_adata.obsm.copy()

In [ ]:
clusters_path = f'{projdir}/csv/clusters/rna_leiden_v7.csv'
clusters = pd.read_csv(clusters_path, sep='\t', index_col=0)
adata.obs['cluster'] = clusters.reindex(adata.obs.index)['clusters_v7']
adata.obs['coarse'] = clusters.reindex(adata.obs.index)['coarse_v7']

In [ ]:
sc.pp.normalize_total(adata)
sc.pp.log1p(adata)

In [ ]:
sc.pl.umap(adata, color=['coarse', 'time_num'])

In [ ]:
oskm = ['POU5F1', 'SOX2', 'KLF4', 'MYC']
indices = adata.var.index.get_indexer(oskm)
indices

In [ ]:
for g,idx in zip(oskm, indices):
    adata.obs[g] = np.ravel(adata[:,idx].X.toarray())

In [ ]:
adata.obs.head()

In [ ]:
whitelist = ['Start', 'Sendai', 'Inter1', 'Delayed', 'Fail1']
mask = np.array([x in whitelist for x in adata.obs['coarse']])
fail_adata = adata[mask]
prod_adata = adata[~mask]
fail_adata

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(10,10))
for i,ax in enumerate(axes.flatten()):
    g = oskm[i]
    sns.boxplot(data=fail_adata.obs, x='cluster', y=g, showfliers=True, ax=ax)
    ax.tick_params(axis='x', rotation=45)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(10,10))
for i,ax in enumerate(axes.flatten()):
    g = oskm[i]
    sns.boxplot(data=fail_adata.obs, x='coarse', y=g, showfliers=True, ax=ax)
    ax.tick_params(axis='x', rotation=45)

In [ ]:
to_plot = fail_adata.obs.melt(id_vars=['coarse'], value_vars=oskm)
to_plot.head()

In [ ]:
sns.boxplot(data=to_plot, x='coarse', y='value', hue='variable',showfliers=True)

In [ ]:
to_plot.head()

In [ ]:
outdir = '/u/project/cluo/terencew/igvf/2023_YR2/final_figures/csv/figure3'
to_plot.to_csv(f'{outdir}/multiome_rna_oskm_fail.csv', sep='\t')

In [ ]:
!date